# Assignment 09 solutions

The traces below follow this reference solution. A correct solution may name its riders differently or put its postcondition assert before the `_check_invariants()` call; what must match is what each method returns, the exceptions it raises, and the links it leaves behind.

---

## Problem 1: A Singly Linked List That Checks Itself

See the assignment notebook for the full problem statement.

**Solution: technical note**

`_check_invariants` is the counting ride from the Linked Lists II deck with one assert at the end. It is worth writing first because everything after it leans on it: once `remove_at` calls it, a wrong `_size` or a lost station stops the program on the line that broke it, not three calls later.

`remove_at` does its work in a fixed order:

1. **Refuse a bad index first.** `index < 0 or index >= self._size` raises `IndexError` before anything changes. `_size` makes this a single comparison, with no ride.
2. **Remember the promise.** `expected = self.get(index)` records the name that should come back, so the postcondition at the end has something to compare against.
3. **Unlink.** Index 0 is the head: `_head` moves to `removed.next`. Any other index rides `index - 1` stops to the station **before** the one being removed, then steps over it with `previous.next = removed.next`.
4. **Keep the counter honest**, then check: `self._size -= 1`, `self._check_invariants()`, and `assert removed.data == expected`.

Trace of `remove_at(2)` on Kimball -> Kedzie -> Francisco -> Rockwell:

| Step | `previous` | What changes |
| --- | --- | --- |
| ride `index - 1 = 1` stop | Kedzie | nothing yet |
| `removed = previous.next` | Kedzie | `removed` is Francisco |
| `previous.next = removed.next` | Kedzie | Kedzie now links to Rockwell |
| `self._size -= 1` | | `_size` goes from 4 to 3 |

The asserts of the form `assert previous is not None, "..."` inside the ride are not the postcondition: they tell the type checker that `previous` is a `Node`, which `_size` guarantees. The postcondition is the last line.

---

## Problem 2: Two Riders

See the assignment notebook for the full problem statement.

**Solution: technical note**

`has_loop` is the deck's `middle` with one more test. The guard `fast is not None and fast.next is not None` is what makes `fast.next.next` safe, and the `and not found` in the same test ends the ride as soon as the riders meet, with no `break`. On a line with no loop, `fast` falls off the end and the answer stays `False`. On a line with a loop, `fast` can never fall off; it gains one stop on `slow` every turn, so once both are inside the loop the gap shrinks by one each turn until it is zero.

On the Brown Line with Rockwell linked back to Kedzie:

| Turn | `slow` | `fast` |
| --- | --- | --- |
| start | Kimball | Kimball |
| 1 | Kedzie | Francisco |
| 2 | Francisco | Kedzie |
| 3 | Rockwell | Rockwell: they meet, `found` is `True` |

Counting cannot replace this. `_size` is 4 on the looped line too: the loop changed a link, not the number of nodes.

---

## Problem 3: Clearing the Loop

See the assignment notebook for the full problem statement.

**Solution: technical note**

The whole method rests on one pointer, `before`, which always sits on the station just before the next one to be counted. At the start that is `_tail`, the deck's "start before the head".

Each round:

1. **Count.** Step `before` forward `k - 1` times. The station after it, `out = before.next`, is the `k`-th one counted.
2. **Unlink.** Record `out.data`. If `out is before`, it is the only station left, and the loop becomes empty (`_tail = None`). Otherwise `before.next = out.next` steps over it, and if `out` was the tail, `_tail` moves back to `before`.
3. **Check.** `self._size -= 1`, then `self._check_invariants()`.

`before` does not move after an unlink: the station after it is now the one after `out`, which is exactly where the next count starts. The `while self._tail is not None` test ends the loop when the last station leaves. The final assert checks the promise that each station left exactly once.

Trace of `eliminate_every(2)` on the Loop:

| Round | `before` after counting | Removed | Notes |
| --- | --- | --- | --- |
| 1 | Clark/Lake | State/Lake | |
| 2 | Washington/Wabash | Adams/Wabash | |
| 3 | Quincy | Clark/Lake | the count went round the end of the loop |
| 4 | Washington/Wabash | Quincy | Quincy was `_tail`, so `_tail` moves back to Washington/Wabash |
| 5 | Washington/Wabash | Washington/Wabash | `out is before`: the only station, so `_tail` becomes `None` |

Two bugs are common here. Forgetting to move `_tail` leaves it on a node that is no longer in the loop, and the given `_check_invariants` reports "the ride never returns to the tail" in that round. Forgetting the one-station case links the last station to itself forever.

---

## Problem 4: A Doubly Linked List

See the assignment notebook for the full problem statement.

**Solution: technical note**

The given `_check_invariants` is what makes both methods checkable: it asserts every `prev` link, the tail and the count, so a swap that is missed at one station fails right there.

`reverse` swaps the two links of every node in one ride. The step that is easy to get wrong is moving on: after the swap, `current.next` points **backward**, so the method saves `following = current.next` before swapping and moves to `following` after. Then the two ends trade places. No node is created and no name moves, so the nodes that were head to tail are now tail to head.

| Station | before the swap (`prev`, `next`) | after the swap (`prev`, `next`) |
| --- | --- | --- |
| Midway | `None`, Pulaski | Pulaski, `None` |
| Pulaski | Midway, Kedzie | Kedzie, Midway |
| Kedzie | Pulaski, Western | Western, Pulaski |
| Western | Kedzie, `None` | `None`, Kedzie |

Then `_head` becomes Western and `_tail` becomes Midway, and the postcondition `assert self._tail is old_first` confirms it.

`remove_last` needs no ride. `removed = self._tail`, and `removed.prev` is the station before it. If there is none, the line had one station and both ends become `None`. Otherwise that station's `next` is cut to `None` and it becomes the tail. That is one or two assignments whatever the line's length, which is the point of the doubly linked list. The postcondition `assert self._tail is None or self._tail.next is None` checks the cut: forgetting it leaves the new tail still pointing at the removed station.

All four problems share one file. `LinkedList` carries Problems 1 and 2, `CircularLinkedList` Problem 3, and `DoublyLinkedList` Problem 4; comment banners mark which problem each method belongs to.

In [ ]:
class Node:
    """One station, linked to the station after it."""

    def __init__(self, data: str) -> None:
        # The station name this node carries
        self._data: str = data
        # The following station; None until it is linked
        self._next: Node | None = None

    @property
    def data(self) -> str:
        """Return the station name."""
        return self._data

    @property
    def next(self) -> "Node | None":
        """Return the following station."""
        return self._next

    @next.setter
    def next(self, node: "Node | None") -> None:
        """Link this station to node."""
        self._next = node


# ----- Problems 1 and 2: a singly linked list with a size counter -----


class LinkedList:
    """A train line: Nodes chained from the head, plus a running count."""

    def __init__(self) -> None:
        # The first station; None means an empty line
        self._head: Node | None = None
        # How many stations are on the line, kept up to date by every change
        self._size: int = 0

    def is_empty(self) -> bool:
        """Return True when the line has no stations."""
        return self._head is None

    def size(self) -> int:
        """Return the number of stations, with no ride."""
        return self._size

    def __str__(self) -> str:
        """Return the stations in order, arrow-joined."""
        names: list[str] = []
        current = self._head
        # Ride until we fall off the end of the line
        while current is not None:
            names.append(current.data)
            current = current.next
        return " -> ".join(names)

    def add_first(self, data: str) -> None:
        """Insert data as the new first station."""
        new_node = Node(data)
        # New station points at the old first station, then becomes the head
        new_node.next = self._head
        self._head = new_node
        self._size += 1

    def add_last(self, data: str) -> None:
        """Append data as the new last station."""
        new_node = Node(data)
        if self._head is None:
            # Empty line: the new station is the whole line
            self._head = new_node
        else:
            # Ride to the last station, then hook the new one on after it
            current = self._head
            while current.next is not None:
                current = current.next
            current.next = new_node
        self._size += 1

    def get(self, index: int) -> str:
        """Return the name of the station at index.

        Precondition: 0 <= index < self.size(), or IndexError is raised.
        """
        # The caller's mistake: raise, so it is reported in every run
        if index < 0 or index >= self._size:
            raise IndexError(f"no station at index {index}")
        current = self._head
        # Ride exactly index stops from the head
        for _ in range(index):
            # Our own promise: _size says this station exists
            assert current is not None, "ran off the end of the line"
            current = current.next
        assert current is not None, "ran off the end of the line"
        return current.data

    # ----- Problem 1 -----

    def _check_invariants(self) -> None:
        """Assert that _size matches the number of nodes on the line.

        Ride the whole line, count the nodes, and assert that the count
        equals self._size. Return nothing when the line is healthy; raise
        AssertionError, with a message, when it is not.
        """
        count = 0
        current = self._head
        # Count every station by riding to the end
        while current is not None:
            count += 1
            current = current.next
        # The counter and the chain of nodes must agree
        assert count == self._size, f"_size is {self._size}, line has {count}"

    def remove_at(self, index: int) -> str:
        """Remove the station at index and return its name.

        Precondition: 0 <= index < self.size(); any other index (including
        every index on an empty line) raises IndexError and leaves the line
        unchanged.
        Postconditions: the returned name was the one at index, and the size
        went down by 1.
        """
        # The caller's mistake: raise before anything changes
        if index < 0 or index >= self._size:
            raise IndexError(f"no station at index {index}")
        # Remember the name at index, to check the promise at the end
        expected = self.get(index)
        if index == 0:
            # Removing the head: the head moves one stop down the line
            removed = self._head
            assert removed is not None, "a non-empty line has a head"
            self._head = removed.next
        else:
            # Ride to the station just before the one we remove
            previous = self._head
            for _ in range(index - 1):
                assert previous is not None, "ran off the end of the line"
                previous = previous.next
            assert previous is not None, "ran off the end of the line"
            removed = previous.next
            assert removed is not None, "ran off the end of the line"
            # Step over the removed station
            previous.next = removed.next
        self._size -= 1
        # Our own promises: the counter matches, and we removed the right one
        self._check_invariants()
        assert removed.data == expected, "removed the wrong station"
        return removed.data

    # ----- Problem 2 -----

    def has_loop(self) -> bool:
        """Return True if following next from the head ever revisits a node,
        False if the ride reaches None.

        Use two riders: one moves one stop per turn, the other two. Do not
        read _size or call size() or len(), and do not change any link.
        """
        found = False
        slow = self._head
        fast = self._head
        # fast rides two stops per turn; if there is a loop it laps slow
        while fast is not None and fast.next is not None and not found:
            assert slow is not None, "slow is never ahead of fast"
            slow = slow.next
            fast = fast.next.next
            found = slow is fast
        return found


# ----- Problem 3: a circular list -----


class CircularLinkedList:
    """A loop of stations: the last station links back to the first.

    The list keeps _tail, not _head; the first station is _tail.next.
    """

    def __init__(self) -> None:
        # The last station; None means an empty loop
        self._tail: Node | None = None
        # How many stations are on the loop
        self._size: int = 0

    def is_empty(self) -> bool:
        """Return True when the loop has no stations."""
        return self._tail is None

    def size(self) -> int:
        """Return the number of stations, with no ride."""
        return self._size

    def __str__(self) -> str:
        """Return the stations once around, first to last, arrow-joined."""
        names: list[str] = []
        if self._tail is not None:
            current = self._tail.next
            # Ride once around: stop after the tail has been added
            for _ in range(self._size):
                assert current is not None, "a loop never reaches None"
                names.append(current.data)
                current = current.next
        return " -> ".join(names)

    def add_first(self, data: str) -> None:
        """Insert data before the first station, keeping the loop closed."""
        new_node = Node(data)
        if self._tail is None:
            # A loop of one points at itself
            self._tail = new_node
            new_node.next = new_node
        else:
            # The new station points at the old first, the tail points at it
            new_node.next = self._tail.next
            self._tail.next = new_node
        self._size += 1

    def add_last(self, data: str) -> None:
        """Append data after the last station: add_first, then move _tail."""
        self.add_first(data)
        assert self._tail is not None, "add_first left the loop empty"
        self._tail = self._tail.next

    def _check_invariants(self) -> None:
        """Given: assert that riding once around from the first station
        returns to it after exactly _size stops.
        """
        count = 0
        if self._tail is not None:
            # Ride from the first station until we are back at the tail;
            # stop early past _size so a broken loop cannot ride forever
            current = self._tail.next
            count = 1
            while (current is not None and current is not self._tail
                   and count <= self._size):
                current = current.next
                count += 1
            assert current is self._tail, "the ride never returns to the tail"
        assert count == self._size, f"_size is {self._size}, loop has {count}"

    # ----- Problem 3 -----

    def eliminate_every(self, k: int) -> list[str]:
        """Empty the loop round-robin and return the names in the order
        they were removed.

        Start counting at the first station, which counts as 1. Remove the
        k-th station counted, then start counting again at the station
        after it, going round the loop as many times as needed, until no
        station is left. With k == 1 the stations leave in order.

        A k below 1 raises ValueError. An empty loop returns []. Unlink each
        station by rewiring the station before it: do not create nodes, do
        not call other methods that change the loop, and do not work the
        order out on a Python list first.
        Postconditions: the loop is empty, and every station was removed
        exactly once.
        """
        # The caller's mistake: there is no 0th station to count to
        if k < 1:
            raise ValueError(f"k must be at least 1, got {k}")
        old_size = self._size
        removed: list[str] = []
        # before always sits just ahead of the next station to count
        before = self._tail
        while self._tail is not None:
            assert before is not None, "a loop never reaches None"
            # Step k - 1 stations, so the k-th is the one after before
            for _ in range(k - 1):
                assert before.next is not None, "a loop never reaches None"
                before = before.next
            out = before.next
            assert out is not None, "a loop never reaches None"
            removed.append(out.data)
            if out is before:
                # The only station left: the loop becomes empty
                self._tail = None
            else:
                # Step over the removed station; a removed tail moves back
                before.next = out.next
                if out is self._tail:
                    self._tail = before
            self._size -= 1
            # Our own promise: still a closed loop after every removal
            self._check_invariants()
        # Our own promise: each station left exactly once
        assert len(removed) == old_size, "a station was skipped or repeated"
        return removed


# ----- Problem 4: a doubly linked list -----


class DoubleNode:
    """One station with links both ways."""

    def __init__(self, data: str) -> None:
        # The station name this node carries
        self._data: str = data
        # The station toward the tail; None at the last station
        self._next: DoubleNode | None = None
        # The station toward the head; None at the first station
        self._prev: DoubleNode | None = None

    @property
    def data(self) -> str:
        """Return the station name."""
        return self._data

    @property
    def next(self) -> "DoubleNode | None":
        """Return the station toward the tail."""
        return self._next

    @next.setter
    def next(self, node: "DoubleNode | None") -> None:
        """Link this station to node, toward the tail."""
        self._next = node

    @property
    def prev(self) -> "DoubleNode | None":
        """Return the station toward the head."""
        return self._prev

    @prev.setter
    def prev(self, node: "DoubleNode | None") -> None:
        """Link this station to node, toward the head."""
        self._prev = node


class DoublyLinkedList:
    """A line that remembers both of its ends."""

    def __init__(self) -> None:
        # The first station; None means an empty line
        self._head: DoubleNode | None = None
        # The last station; None means an empty line
        self._tail: DoubleNode | None = None
        # How many stations are on the line
        self._size: int = 0

    def is_empty(self) -> bool:
        """Return True when the line has no stations."""
        return self._head is None

    def size(self) -> int:
        """Return the number of stations, with no ride."""
        return self._size

    def __str__(self) -> str:
        """Return the stations head to tail, arrow-joined."""
        names: list[str] = []
        current = self._head
        # Ride toward the tail until we fall off the end
        while current is not None:
            names.append(current.data)
            current = current.next
        return " -> ".join(names)

    def add_last(self, data: str) -> None:
        """Append a station after the current tail, with no ride."""
        new_node = DoubleNode(data)
        if self._tail is None:
            # Empty line: the new station is both ends
            self._head = new_node
        else:
            # The new station points back first, then the old tail points forward
            new_node.prev = self._tail
            self._tail.next = new_node
        self._tail = new_node
        self._size += 1

    def _check_invariants(self) -> None:
        """Given: assert that the line agrees with itself.

        Ride from _head toward the tail and assert three things: each
        node's prev is the node the ride just left (None for the head);
        the last node the ride visits is _tail (None on an empty line); and
        the number of nodes equals _size.
        """
        previous = None
        current = self._head
        count = 0
        # Ride toward the tail, checking each backward link on the way
        while current is not None:
            assert current.prev is previous, f"bad prev at {current.data!r}"
            previous = current
            current = current.next
            count += 1
        # The ride must end on the tail, after exactly _size stations
        assert previous is self._tail, "_tail is not the last station"
        assert count == self._size, f"_size is {self._size}, line has {count}"

    # ----- Problem 4 -----

    def reverse(self) -> None:
        """Reverse the line in place: the old tail becomes the head.

        Create no nodes and change no node's data: swap each node's next
        and prev links, then swap _head and _tail. An empty line stays
        empty.
        Postcondition: the old first station is now the last one.
        """
        old_first = self._head
        current = self._head
        # Ride toward the old tail, swapping both links at every station
        while current is not None:
            following = current.next  # save the way on before flipping
            current.next = current.prev
            current.prev = following
            current = following
        # The two ends trade places
        old_head = self._head
        self._head = self._tail
        self._tail = old_head
        # Our own promises: every link agrees, and the old first is now last
        self._check_invariants()
        assert self._tail is old_first, "the old first station is not last"

    def remove_last(self) -> str:
        """Remove the last station and return its name.

        Raise IndexError on an empty line (lecture's version returned None;
        this one fails loudly, like list.pop). Removing the only station
        leaves an empty line. No loop: the tail already knows the station
        before it, so the rewiring takes no ride. (The _check_invariants
        call at the end does ride the line; that is the price of checking.)
        Postcondition: the new tail, if any, links to nothing after it.
        """
        # The caller's mistake: nothing to remove
        if self._tail is None:
            raise IndexError("remove_last from an empty line")
        removed = self._tail
        if removed.prev is None:
            # The only station: both ends become None
            self._head = None
            self._tail = None
        else:
            # The station before becomes the last one
            removed.prev.next = None
            self._tail = removed.prev
        self._size -= 1
        # Our own promises: every link agrees, and nothing follows the new tail
        self._check_invariants()
        assert self._tail is None or self._tail.next is None, "tail links onward"
        return removed.data


**Test cases**

In [ ]:
# Each problem's checks run on their own, so you can work in any order:
# an unfinished or crashing problem prints one line and the rest still run.

def error_name(action) -> str:
    """Run action() and return the name of the exception it raises."""
    name = "no error"
    try:
        action()
    except Exception as error:  # noqa: BLE001 -- report any exception by name
        name = type(error).__name__
    return name

def brown_line() -> LinkedList:
    """Build a four-station Brown Line, north to south."""
    line = LinkedList()
    for station in ["Kimball", "Kedzie", "Francisco", "Rockwell"]:
        line.add_last(station)
    return line

def problem_1() -> None:
    # ----- Problem 1 -----
    line = brown_line()
    line._check_invariants()                # a healthy line: nothing printed
    print(line.remove_at(2), "|", line)     # expected: Francisco | Kimball -> Kedzie -> Rockwell
    print(line.remove_at(0), line.remove_at(1), "|", line, line.size())
    # expected: Kimball Rockwell | Kedzie 1
    # A bad index is the caller's mistake: IndexError, line unchanged.
    print(error_name(lambda: line.remove_at(1)),
          error_name(lambda: line.remove_at(-1)),
          error_name(lambda: LinkedList().remove_at(0)))
    # expected: IndexError IndexError IndexError
    print(line, line.size())                # expected: Kedzie 1
    # Break the counter on purpose (code outside the class should never
    # touch _size): _check_invariants must catch it.
    line._size = 7
    print(error_name(line._check_invariants))   # expected: AssertionError

def problem_2() -> None:
    # ----- Problem 2 -----
    line = brown_line()
    print(line.has_loop(), LinkedList().has_loop())     # expected: False False
    # Link the last station back to the second, on purpose, to make a loop.
    # Do not print a line with a loop: __str__ would ride forever.
    # Each assert tells the type checker a node exists, as it does in the class.
    assert line._head is not None, "the line has four stations"
    second = line._head.next
    assert second is not None and second.next is not None, "four stations"
    last = second.next.next
    assert last is not None, "the line has four stations"
    last.next = second
    print(line.has_loop())                  # expected: True

def problem_3() -> None:
    # ----- Problem 3 -----
    def the_loop() -> CircularLinkedList:
        loop = CircularLinkedList()
        for station in ["Clark/Lake", "State/Lake", "Washington/Wabash",
                        "Adams/Wabash", "Quincy"]:
            loop.add_last(station)
        return loop

    loop = the_loop()
    print(loop)                             # expected: Clark/Lake -> State/Lake -> Washington/Wabash -> Adams/Wabash -> Quincy
    print(loop.eliminate_every(2))
    # expected: ['State/Lake', 'Adams/Wabash', 'Clark/Lake', 'Quincy', 'Washington/Wabash']
    print(loop.is_empty(), loop.size(), repr(str(loop)))     # expected: True 0 ''
    print(the_loop().eliminate_every(1))
    # expected: ['Clark/Lake', 'State/Lake', 'Washington/Wabash', 'Adams/Wabash', 'Quincy']
    print(the_loop().eliminate_every(7))
    # expected: ['State/Lake', 'Quincy', 'Clark/Lake', 'Washington/Wabash', 'Adams/Wabash']
    print(CircularLinkedList().eliminate_every(3),
          error_name(lambda: the_loop().eliminate_every(0)))
    # expected: [] ValueError

def problem_4() -> None:
    # ----- Problem 4 -----
    orange = DoublyLinkedList()
    for station in ["Midway", "Pulaski", "Kedzie", "Western", "35th/Archer"]:
        orange.add_last(station)
    orange._check_invariants()              # a healthy line: nothing printed
    print(orange.remove_last(), orange)     # expected: 35th/Archer Midway -> Pulaski -> Kedzie -> Western
    orange.reverse()
    print(orange)                           # expected: Western -> Kedzie -> Pulaski -> Midway
    print(orange.remove_last(), orange)     # expected: Midway Western -> Kedzie -> Pulaski
    orange.reverse()
    orange.add_last("Halsted")
    print(orange)                           # expected: Pulaski -> Kedzie -> Western -> Halsted
    for _ in range(4):
        orange.remove_last()
    orange.reverse()
    print(orange.is_empty(), error_name(orange.remove_last))   # expected: True IndexError
    for station in ["Midway", "Pulaski", "Kedzie"]:
        orange.add_last(station)
    # Cut one backward link on purpose: _check_invariants must catch it.
    assert orange._tail is not None, "the line has three stations"
    orange._tail.prev = orange._head
    print(error_name(orange._check_invariants))         # expected: AssertionError

for check in (problem_1, problem_2, problem_3, problem_4):
    try:
        check()
    except NotImplementedError:
        print(f"{check.__name__}: not finished yet, skipped")
    except Exception as error:  # noqa: BLE001 -- report any crash, keep going
        print(f"{check.__name__}: stopped by {type(error).__name__}: {error}")
